# Translate French, English, and Ngiemboon (guembon)

**Ngiemboon** (also spelled *ngienboum*, *guembon*) is a Grassfields language spoken in western Cameroon, especially in **Batcham** (Bamboutos). The 2007 New Testament spelling follows the **General Alphabet of Cameroon Languages**: ŋ, ɛ, ɔ, ʉ, ẅ, and tones. They are letters, not noise.

Sources are listed in `data/README.md`. The instructions are in `README.md`. The aligned New Testament is most of the corpus. The health voice-over (Timene 2024) adds spoken French and medical vocabulary. The Anderson sketch (SIL) fixes the AGLC spelling. Ndiola 2011 is for reading the syntax, not for the training spelling.

English is not in the training files. At translation time it goes through French (Helsinki-NLP Opus-MT), then through this model.

| File | Language | Coverage |
| --- | --- | --- |
| `Francais-Bible-Segond-1910-All-Bible.pdf` | French (Segond 1910) | whole Bible |
| `Ngiemboon-Bible-New-Testasment.pdf` | Ngiemboon (Bible Society of Cameroon, 2007) | **New Testament only** |

Only the **New Testament** is aligned. A French verse and a Ngiemboon verse with the same reference form one training pair.

One Transformer learns **both directions**: the source is prefixed with `<2ng>` (into Ngiemboon) or `<2fr>` (into French).

This is not a finished product. About eight thousand verses are still too few for a large network. The model is **small** on purpose. Adding pairs means adding lines to `processed/pairs.jsonl` and training again.


In [ ]:
from pathlib import Path
import sys

here = Path.cwd()
root = here if (here / "nmt").exists() else here / "Transformer"
sys.path.insert(0, str(root))
print("project root:", root)


## 1. Extract and align the verses

Each PDF page is text, not a scan. Headers, page numbers, and cross-references such as `V. 1-17: cf. …` or `(Lukase 3.23-38)` are removed, then the text is split into:

1. a **book + chapter** (`Matthieu 3`, `Matio 3`, `Megàŋ ntúm 2`, `1 Korento 13`);
2. **verses**: a number glued to a letter.

The alignment key is canonical: `JHN.3.16`. A verse that exists on only one side is left out.


In [ ]:
import json
from nmt.parse_bibles import PROCESSED_DIR, build_corpus

pairs = build_corpus(force=False)
stats = json.loads((PROCESSED_DIR / "stats.json").read_text(encoding="utf-8"))
print(json.dumps(stats, ensure_ascii=False, indent=2))
print()
for row in pairs:
    if row["id"] in {"MAT.1.1", "JHN.3.16"}:
        print(row["ref"])
        print(" FR:", row["fr"])
        print(" NG:", row["ng"])
        print()


## 2. One shared vocabulary (BPE)

French and Ngiemboon share some proper names but not the morphology. A **joint BPE** learns pieces useful to both sides **without breaking** AGLC characters: `ŋ`, `ẅ`, and a tone stay atomic symbols.

| Token | Role |
| --- | --- |
| `<pad>` | fills a batch |
| `<bos>` / `<eos>` | start / end of a sequence |
| `<2ng>` | translate into Ngiemboon |
| `<2fr>` | translate into French |


In [ ]:
from nmt.train import prepare

all_pairs, train_pairs, val_pairs, tokenizer = prepare(force_extract=False)
print("train verses", len(train_pairs), "val verses", len(val_pairs))
print("vocab", tokenizer.vocab_size)
print(tokenizer.encode("Car Dieu a tant aimé le monde", lang_tag="ng")[:20])
print(tokenizer.decode(tokenizer.encode("Car Dieu a tant aimé le monde", lang_tag="ng")))


## 3. The model

Pre-norm Transformer: multi-head attention, GELU, residuals, LayerNorm, a causal mask, tied embeddings. Without a GPU: 3 layers, `d_model = 192`. With CUDA: 4 layers, `d_model = 256`.

In training, each verse becomes **two** examples:

```text
<2ng> + French       →  Ngiemboon
<2fr> + Ngiemboon    →  French
```

From the repository root: `python -m nmt.corpus --force`, then `python -m nmt train`. The next cell loads a checkpoint if one exists.


In [ ]:
from nmt.paths import CHECKPOINTS
from nmt.train import load_model

ckpt = CHECKPOINTS / "fr_ngiemboon.pt"
if ckpt.exists():
    model, tokenizer, device = load_model()
    print("loaded", ckpt, "on", device)
else:
    print("No checkpoint. Run: python -m nmt train")


## 4. Translate both ways

Greedy decoding takes the most probable token at each step, until `<eos>`. Beam search is in `nmt/decode.py`. The command line uses a beam of 4.

New Testament verses the model has seen should already look like something. An everyday sentence is more fragile, because the corpus is mostly biblical.

English does not go through `translate_pair` directly. `translate_between` in `nmt.pivot` turns English into French first.


In [ ]:
from nmt.decode import translate_pair

examples_fr = [
    "Généalogie de Jésus-Christ, fils de David, fils d’Abraham",
    "Car Dieu a tant aimé le monde qu’il a donné son Fils unique, afin que quiconque croit en lui ne périsse point, mais qu’il ait la vie éternelle",
    "Je suis le chemin, la vérité, et la vie",
]

for text in examples_fr:
    hyp = translate_pair(model, tokenizer, text, "fr2ng", device)
    print("FR:", text)
    print("NG:", hyp)
    print()


In [ ]:
examples_ng = [
    "Mezíŋ pàmangʉa njʉ̀’ Yěsô Kristo gẅie aa na ngẅɛ fùɔ Ndavíd, éfóo na ngẅɛ Ábalam e mɔɔn",
    "Á nue ntí gie e Ssé la lɔg ńkwoŋ njÿó pú’u la gwɔ́ tà á náa Míŋ we wémɔ’ɔ́, légwɔ́ gie wɛ̌ nyìŋ wɔ́ gẅie à zǎb ntʉ́m wé ná Míŋ we ŋwɛ́, à tě ńdá’a ńkaa ndòon tèmì wɔ́, à ge lá’ gẅiin le gwɔ́ tèmì",
]

for text in examples_ng:
    hyp = translate_pair(model, tokenizer, text, "ng2fr", device)
    print("NG:", text)
    print("FR:", hyp)
    print()


## 5. Your own sentence

Change `phrase` and `direction`, then run the cell again. `en2ng` and `ng2en` use the French pivot.


In [ ]:
from nmt.pivot import translate_between

phrase = "Que la grâce et la paix vous soient données"
direction = "fr2ng"  # or "ng2fr", "en2ng", "ng2en"

text, notes = translate_between(model, tokenizer, phrase, direction, device)
print(text)
for note in notes:
    print("Note:", note)


## 6. Command line

From the repository root, once a checkpoint exists:

```bash
python -m nmt "Bonsoir, comment ça va ?"
python -m nmt --from en "Good evening"
python -m nmt --to fr "má"
python -m nmt --to en "má"
```

To train again after adding pairs (same JSONL fields: `id`, `fr`, `ng`):

```bash
python -m nmt train --rebuild-corpus
```

| Path | Role |
| --- | --- |
| `nmt/parse_bibles.py` | PDF to aligned verses |
| `nmt/tokenizer.py` | joint BPE |
| `nmt/model.py` | Transformer |
| `nmt/train.py` | training loop |
| `nmt/decode.py` / `nmt/pivot.py` | inference, including English through French |
| `processed/pairs.jsonl` | corpus |
| `checkpoints/fr_ngiemboon.pt` | weights |

The Ngiemboon New Testament is © Alliance biblique du Cameroun, 2007. Segond 1910 is in the public domain. Do not republish the aligned corpus.

Window: `python -m nmt app` (it does not train). The window switches between English and French. Training: `python -m nmt train`.
